<a href="https://colab.research.google.com/github/eunchae0215/game/blob/main/%5BBDA%5D%EC%98%88%EC%A0%9C_%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install flask_cors flask pyngrok

In [ ]:
# ====== 2. 라이브러리 임포트 ======
from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import threading
from pyngrok import ngrok
import warnings
warnings.filterwarnings('ignore')


# ngrok 계정이 없다면 https://dashboard.ngrok.com/signup 에서 가입
# 가입 후 authtoken을 받아서 아래에 입력
ngrok.set_auth_token("2lEWDPFPQButa0K0tJcFYlWh6eV_4sCYk7CKk7YsWkKgkueRt")  # 실제 토큰으로 교체
# ====== 3. 집값 예측 모델 학습 ======
print("집값 예측 모델 학습 중...")

# California Housing 데이터 로드
housing = fetch_california_housing()
X = housing.data
y = housing.target  # 집값 (단위: 십만 달러)

# 특성 이름
feature_names = [
    '중간 소득',
    '주택 연령',
    '평균 방수',
    '평균 침실수',
    '인구',
    '평균 거주자수',
    '위도',
    '경도'
]

# 학습/테스트 분할 (정규화 제거 - 간소화)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest 모델 학습
model = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# 성능 평가
test_score = model.score(X_test, y_test)
print(f"모델 학습 완료! 테스트 R² 점수: {test_score:.3f}")


집값 예측 모델 학습 중...
모델 학습 완료! 테스트 R² 점수: 0.773


In [ ]:
# ====== 4. HTML 템플릿 ======
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>캘리포니아 집값 예측기</title>
</head>
<body>
    <h1>캘리포니아 집값 예측기</h1>
    <p>Random Forest 모델을 사용한 실시간 예측</p>

    <form id="predictionForm">
        <div>
            <label for="MedInc">중간 소득 (만 달러):</label>
            <input type="number" id="MedInc" step="0.1" value="3.5" required>
        </div>

        <div>
            <label for="HouseAge">주택 연령 (년):</label>
            <input type="number" id="HouseAge" step="1" value="20" required>
        </div>

        <div>
            <label for="AveRooms">평균 방 개수:</label>
            <input type="number" id="AveRooms" step="0.1" value="5.5" required>
        </div>

        <div>
            <label for="AveBedrms">평균 침실 수:</label>
            <input type="number" id="AveBedrms" step="0.1" value="1.0" required>
        </div>

        <div>
            <label for="Population">인구:</label>
            <input type="number" id="Population" step="10" value="3000" required>
        </div>

        <div>
            <label for="AveOccup">평균 거주자 수:</label>
            <input type="number" id="AveOccup" step="0.1" value="3.0" required>
        </div>

        <div>
            <label for="Latitude">위도:</label>
            <input type="number" id="Latitude" step="0.01" value="34.0" required>
        </div>

        <div>
            <label for="Longitude">경도:</label>
            <input type="number" id="Longitude" step="0.01" value="-118.0" required>
        </div>

        <button type="submit">집값 예측하기</button>
    </form>

    <div id="result"></div>
    <div id="error"></div>

    <script>
        document.getElementById('predictionForm').addEventListener('submit', async (e) => {
            e.preventDefault();

            // 결과 및 에러 메시지 초기화
            document.getElementById('result').innerHTML = '';
            document.getElementById('error').innerHTML = '';

            // 입력값 수집
            const features = [
                parseFloat(document.getElementById('MedInc').value),
                parseFloat(document.getElementById('HouseAge').value),
                parseFloat(document.getElementById('AveRooms').value),
                parseFloat(document.getElementById('AveBedrms').value),
                parseFloat(document.getElementById('Population').value),
                parseFloat(document.getElementById('AveOccup').value),
                parseFloat(document.getElementById('Latitude').value),
                parseFloat(document.getElementById('Longitude').value)
            ];

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({features: features})
                });

                const result = await response.json();

                if (result.error) {
                    throw new Error(result.error);
                }

                // 결과 표시
                document.getElementById('result').innerHTML =
                    `<h2>예측 결과</h2>
                     <p>예측 가격: $${result.price.toLocaleString()}</p>`;

            } catch (error) {
                document.getElementById('error').innerHTML =
                    `<p>오류: ${error.message}</p>`;
            }
        });
    </script>
</body>
</html>
'''

In [ ]:
# ====== 4. HTML 템플릿 ======
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>🏠 캘리포니아 집값 예측기</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #43cea2 0%, #185a9d 100%);
            min-height: 100vh;
            padding: 20px;
        }

        .container {
            max-width: 600px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            padding: 40px;
        }

        h1 {
            color: #333;
            text-align: center;
            margin-bottom: 10px;
        }

        .subtitle {
            text-align: center;
            color: #666;
            margin-bottom: 30px;
            font-size: 14px;
        }

        .input-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            margin-bottom: 20px;
        }

        .input-group {
            margin-bottom: 15px;
        }

        label {
            display: block;
            color: #555;
            margin-bottom: 5px;
            font-weight: 500;
            font-size: 14px;
        }

        input[type="number"] {
            width: 100%;
            padding: 10px;
            border: 2px solid #e0e0e0;
            border-radius: 8px;
            font-size: 14px;
            transition: border-color 0.3s;
        }

        input[type="number"]:focus {
            outline: none;
            border-color: #43cea2;
        }

        .hint {
            font-size: 11px;
            color: #888;
            margin-top: 2px;
        }

        button {
            width: 100%;
            padding: 15px;
            background: linear-gradient(135deg, #43cea2 0%, #185a9d 100%);
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 18px;
            font-weight: 600;
            cursor: pointer;
            transition: transform 0.2s;
            margin-top: 20px;
        }

        button:hover {
            transform: translateY(-2px);
        }

        button:disabled {
            opacity: 0.6;
            cursor: not-allowed;
        }

        #result {
            margin-top: 30px;
            padding: 25px;
            background: #f0f8ff;
            border-radius: 12px;
            display: none;
            text-align: center;
        }

        #result.show {
            display: block;
            animation: slideIn 0.5s;
        }

        @keyframes slideIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .price-prediction {
            font-size: 36px;
            font-weight: bold;
            color: #185a9d;
            margin-bottom: 10px;
        }

        .error-message {
            color: #e74c3c;
            text-align: center;
            margin-top: 20px;
            padding: 10px;
            background: #ffe6e6;
            border-radius: 8px;
            display: none;
        }

        .error-message.show {
            display: block;
        }

        .loading {
            display: none;
            text-align: center;
            color: #666;
        }

        .loading.show {
            display: block;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🏠 캘리포니아 집값 예측기</h1>
        <div class="subtitle">Random Forest 모델을 사용한 실시간 예측</div>

        <form id="predictionForm">
            <div class="input-grid">
                <div class="input-group">
                    <label for="MedInc">중간 소득</label>
                    <input type="number" id="MedInc" step="0.1" value="3.5" required>
                    <div class="hint">지역 중간 소득 (만 달러)</div>
                </div>

                <div class="input-group">
                    <label for="HouseAge">주택 연령</label>
                    <input type="number" id="HouseAge" step="1" value="20" required>
                    <div class="hint">평균 주택 연령 (년)</div>
                </div>

                <div class="input-group">
                    <label for="AveRooms">평균 방 개수</label>
                    <input type="number" id="AveRooms" step="0.1" value="5.5" required>
                    <div class="hint">가구당 평균 방 수</div>
                </div>

                <div class="input-group">
                    <label for="AveBedrms">평균 침실 수</label>
                    <input type="number" id="AveBedrms" step="0.1" value="1.0" required>
                    <div class="hint">가구당 평균 침실 수</div>
                </div>

                <div class="input-group">
                    <label for="Population">인구</label>
                    <input type="number" id="Population" step="10" value="3000" required>
                    <div class="hint">블록 그룹 인구</div>
                </div>

                <div class="input-group">
                    <label for="AveOccup">평균 거주자 수</label>
                    <input type="number" id="AveOccup" step="0.1" value="3.0" required>
                    <div class="hint">가구당 평균 거주자</div>
                </div>

                <div class="input-group">
                    <label for="Latitude">위도</label>
                    <input type="number" id="Latitude" step="0.01" value="34.0" required>
                    <div class="hint">북위 (도)</div>
                </div>

                <div class="input-group">
                    <label for="Longitude">경도</label>
                    <input type="number" id="Longitude" step="0.01" value="-118.0" required>
                    <div class="hint">서경 (도)</div>
                </div>
            </div>

            <button type="submit" id="predictButton">집값 예측하기</button>
        </form>

        <div class="loading" id="loading">예측 중...</div>

        <div id="result">
            <div class="price-prediction" id="predictedPrice"></div>
        </div>

        <div class="error-message" id="errorMessage"></div>
    </div>

    <script>
        document.getElementById('predictionForm').addEventListener('submit', async (e) => {
            e.preventDefault();

            // UI 상태 변경
            const button = document.getElementById('predictButton');
            const loading = document.getElementById('loading');
            const errorDiv = document.getElementById('errorMessage');
            const resultDiv = document.getElementById('result');

            button.disabled = true;
            loading.classList.add('show');
            errorDiv.classList.remove('show');
            resultDiv.classList.remove('show');

            // 입력값 수집
            const features = [
                parseFloat(document.getElementById('MedInc').value),
                parseFloat(document.getElementById('HouseAge').value),
                parseFloat(document.getElementById('AveRooms').value),
                parseFloat(document.getElementById('AveBedrms').value),
                parseFloat(document.getElementById('Population').value),
                parseFloat(document.getElementById('AveOccup').value),
                parseFloat(document.getElementById('Latitude').value),
                parseFloat(document.getElementById('Longitude').value)
            ];

            console.log('전송할 데이터:', features);

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({features: features})
                });

                const result = await response.json();
                console.log('받은 결과:', result);

                if (result.error) {
                    throw new Error(result.error);
                }

                // 결과 표시
                document.getElementById('predictedPrice').textContent =
                    `예측 가격: $${result.price.toLocaleString()}`;
                resultDiv.classList.add('show');

            } catch (error) {
                console.error('에러:', error);
                errorDiv.textContent = '예측 중 오류가 발생했습니다: ' + error.message;
                errorDiv.classList.add('show');
            } finally {
                button.disabled = false;
                loading.classList.remove('show');
            }
        });
    </script>
</body>
</html>
'''


In [ ]:
# ====== 5. Flask 앱 생성 ======
app = Flask(__name__)
CORS(app)

@app.route('/')
def home():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # 입력 데이터 받기
        data = request.json
        print(f"받은 데이터: {data}")

        # features 배열 추출
        if 'features' not in data:
            return jsonify({'error': 'features 데이터가 없습니다'}), 400

        features = data['features']

        # 8개 특성이 모두 있는지 확인
        if len(features) != 8:
            return jsonify({'error': f'8개의 특성이 필요합니다. 받은 개수: {len(features)}'}), 400

        # numpy 배열로 변환
        features_array = np.array(features).reshape(1, -1)
        print(f"특성 배열 shape: {features_array.shape}")

        # 예측
        prediction = model.predict(features_array)[0]
        print(f"예측값: {prediction}")

        # 결과를 달러로 변환 (십만 달러 → 달러)
        price_in_dollars = int(prediction * 100000)

        # 결과 반환
        result = {
            'price': price_in_dollars,
            'success': True
        }

        print(f"반환할 결과: {result}")
        return jsonify(result)

    except Exception as e:
        print(f"서버 에러: {str(e)}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

In [ ]:
# ====== 6. 서버 실행 ======
def run_app():
    app.run(port=5000)

# Flask를 백그라운드에서 실행
threading.Thread(target=run_app, daemon=True).start()

# ngrok 터널 생성
print("\n🏠 집값 예측 서버 시작 중...")
public_url = ngrok.connect(5000)
print(f"\n✅ 웹 애플리케이션이 실행되었습니다!")
print(f"🌐 공개 URL: {public_url}")
print(f"\n위 URL을 클릭하여 집값 예측기를 사용하세요!")
print("\n종료하려면 런타임을 중지하세요.")

# 서버 유지
import time
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n서버를 종료합니다...")
    ngrok.kill()


🏠 집값 예측 서버 시작 중...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



✅ 웹 애플리케이션이 실행되었습니다!
🌐 공개 URL: NgrokTunnel: "https://ca940a8bb5da.ngrok-free.app" -> "http://localhost:5000"

위 URL을 클릭하여 집값 예측기를 사용하세요!

종료하려면 런타임을 중지하세요.


INFO:werkzeug:127.0.0.1 - - [26/Aug/2025 10:15:21] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Aug/2025 10:15:21] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [26/Aug/2025 10:15:22] "POST /predict HTTP/1.1" 200 -


받은 데이터: {'features': [3.5, 20, 5.5, 1, 3000, 3, 34, -118]}
특성 배열 shape: (1, 8)
예측값: 1.9735600941830602
반환할 결과: {'price': 197356, 'success': True}

서버를 종료합니다...
